# AdaptiveSpectralGuard — Stabilized V2, fail-fast 30-epoch run

This notebook **cannot silently fall back to the original V1 controller**.

Before training it:

1. removes every cached `adaptive_spectral_guard` module from the Jupyter kernel;
2. imports the package directly from this repository clone;
3. requires `STABILIZED_V2_API >= 2`;
4. builds the exact Stabilized V2 configuration in package code;
5. runs a synthetic low-confidence, \(\alpha<2\) preflight;
6. refuses to train unless trace-log volume protection remains active while the uncertain \(\beta_E\) channel is vetoed.

During training, every epoch must print:

```text
STABILIZED V2 CHANNEL GATES FOR NEXT EPOCH
```

The run terminates with an error if it detects the old `off / low ECS confidence` behavior for an enabled layer below the alpha boundary.


In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import weightwatcher as ww
from IPython.display import display

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "adaptive_spectral_guard").is_dir():
        ROOT = candidate
        break
    nested = candidate / "optimizers" / "adaptive_spectral_guard"
    if (nested / "adaptive_spectral_guard").is_dir():
        ROOT = nested
        break

if ROOT is None:
    raise RuntimeError("Open this notebook from a clone of rg_optimizers.")

ROOT = ROOT.resolve()

# A normal `importlib.reload` is insufficient because submodules retain old
# class definitions. Remove the entire package tree from the kernel.
for module_name in list(sys.modules):
    if (
        module_name == "adaptive_spectral_guard"
        or module_name.startswith("adaptive_spectral_guard.")
    ):
        del sys.modules[module_name]

sys.path = [entry for entry in sys.path if Path(entry or ".").resolve() != ROOT]
sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()

import adaptive_spectral_guard as asg

PACKAGE_FILE = Path(asg.__file__).resolve()
if ROOT not in PACKAGE_FILE.parents:
    raise RuntimeError(
        "Imported adaptive_spectral_guard from the wrong location: "
        f"{PACKAGE_FILE}; expected a file under {ROOT}"
    )
if getattr(asg, "STABILIZED_V2_API", 0) < 2:
    raise RuntimeError(
        "Stale AdaptiveSpectralGuard package detected. Pull main and rerun "
        "this cell; STABILIZED_V2_API >= 2 is required."
    )

REPO_ROOT = ROOT.parents[1]
try:
    GIT_HEAD = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
        text=True,
    ).strip()
except Exception:
    GIT_HEAD = "unknown"

print("Repository root:", REPO_ROOT)
print("Package root:", ROOT)
print("Imported package:", PACKAGE_FILE)
print("Package version:", asg.__version__)
print("Stabilized V2 API:", asg.STABILIZED_V2_API)
print("Git HEAD:", GIT_HEAD)
print("Torch:", torch.__version__)
print("WeightWatcher:", getattr(ww, "__version__", "unknown"))


In [ ]:
from adaptive_spectral_guard import (
    assert_stabilized_v2_controller_frame,
    build_stabilized_v2_configuration,
    run_stabilized_v2_mnist,
    run_stabilized_v2_preflight,
    stabilized_v2_policy_table,
)

configuration = build_stabilized_v2_configuration(
    epochs=30,
    seed=1337,
)

print("LAYER POLICIES")
display(stabilized_v2_policy_table(configuration.guard))

print("CONTROLLER CONFIGURATION")
display(
    pd.Series(
        vars(configuration.guard.controller),
        name="Stabilized V2",
    ).to_frame()
)

print("FAIL-FAST LOW-CONFIDENCE / ALPHA<2 PREFLIGHT")
preflight = run_stabilized_v2_preflight(configuration.guard)
display(
    preflight[
        [
            "parameter",
            "regime",
            "reason",
            "alpha",
            "raw_confidence",
            "smoothed_confidence",
            "volume_confidence",
            "shape_confidence",
            "volume_effective_gain",
            "shape_effective_gain",
            "shape_active",
        ]
    ]
)

fc1_probe = preflight.loc[preflight["parameter"].eq("fc1.weight")].iloc[0]
assert fc1_probe["regime"] == "strong"
assert float(fc1_probe["volume_effective_gain"]) > 0.0
assert float(fc1_probe["shape_effective_gain"]) == 0.0
assert fc1_probe["reason"] != "low ECS confidence"

print("PREFLIGHT PASSED: this kernel is running Stabilized V2, not V1.")


## Run the paired experiment

The package wrapper repeats the V2 preflight immediately before training and validates every controller frame printed during the run.

A clean stop is still available through the printed `STOP_AFTER_CURRENT_EPOCH` file. A `KeyboardInterrupt` preserves completed epochs.


In [ ]:
RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = (
    ROOT
    / f"results_adaptive_spectral_guard_stabilized_v2_30epochs_{RUN_STAMP}"
)

print("Output directory:", OUTPUT_DIR.resolve())

result = run_stabilized_v2_mnist(
    configuration,
    data_dir=ROOT / "data",
    output_dir=OUTPUT_DIR,
    progress=True,
)

print("Completed through epoch:", int(result.performance["epoch"].max()))
print("Saved to:", result.output_dir.resolve())


## Performance and WeightWatcher trajectories

In [ ]:
from adaptive_spectral_guard.plotting import (
    plot_controller,
    plot_corrections,
    plot_matched_convergence,
    plot_performance,
    plot_weightwatcher,
)

plot_performance(result.performance)

plot_weightwatcher(
    result.weightwatcher,
    "alpha",
    reference=2.0,
    title=r"All layers: WeightWatcher $\alpha$",
    ylabel=r"WeightWatcher $\alpha$",
)

plot_weightwatcher(
    result.weightwatcher,
    "ERG_gap",
    reference=0.0,
    title="All layers: WeightWatcher ERG gap",
    ylabel="WeightWatcher ERG gap",
)

plot_weightwatcher(
    result.weightwatcher,
    "beta_E_midpoint",
    reference=0.0,
    title=r"All layers: shell $\beta_E$",
    ylabel=r"Shell $\beta_E$",
)


## Verify that the V2 controller actually ran

This cell rejects the exact failure previously observed: an enabled FC1 or FC2 layer below the boundary being reported as `off` because of `low ECS confidence`.


In [ ]:
controller = result.controller.copy()
assert_stabilized_v2_controller_frame(
    controller.loc[controller["epoch"].ge(1)],
    configuration.guard,
)

required_controller_columns = {
    "raw_confidence",
    "smoothed_confidence",
    "volume_confidence",
    "shape_confidence",
    "volume_effective_gain",
    "shape_effective_gain",
}
missing = required_controller_columns - set(controller.columns)
if missing:
    raise RuntimeError(f"Missing V2 controller columns: {sorted(missing)}")

show_controller = controller.loc[
    controller["epoch"].ge(1),
    [
        "epoch",
        "parameter",
        "regime",
        "reason",
        "alpha",
        "raw_confidence",
        "smoothed_confidence",
        "volume_confidence",
        "shape_confidence",
        "volume_effective_gain",
        "shape_effective_gain",
        "shape_active",
    ],
]
display(show_controller.tail(18))

for parameter in ("fc1.weight", "fc2.weight"):
    layer = show_controller.loc[show_controller["parameter"].eq(parameter)]
    fig, ax = plt.subplots(figsize=(10, 5), dpi=135)
    ax.plot(
        layer["epoch"],
        layer["raw_confidence"],
        marker="o",
        label="Raw ECS confidence",
    )
    ax.plot(
        layer["epoch"],
        layer["smoothed_confidence"],
        marker="o",
        label="Smoothed ECS confidence",
    )
    ax.plot(
        layer["epoch"],
        layer["volume_confidence"],
        marker="o",
        label="Trace-log volume confidence",
    )
    ax.plot(
        layer["epoch"],
        layer["shape_confidence"],
        marker="o",
        label=r"Shape / $\beta_E$ confidence",
    )
    ax.set(
        xlabel="Epoch",
        ylabel="Confidence",
        title=f"{parameter}: Stabilized V2 confidence channels",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5), dpi=135)
    ax.plot(
        layer["epoch"],
        layer["volume_effective_gain"],
        marker="o",
        label="Trace-log volume gain",
    )
    ax.plot(
        layer["epoch"],
        layer["shape_effective_gain"],
        marker="o",
        label=r"Shape / $\beta_E$ gain",
    )
    ax.set(
        xlabel="Epoch",
        ylabel="Effective gain",
        title=f"{parameter}: channel-specific gains",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


## Correction saturation and task-conflict checks

In [ ]:
steps = result.guard_steps.copy()
if steps.empty:
    print("No correction steps were recorded.")
else:
    for column in (
        "shape_correction_ratio",
        "volume_correction_ratio",
        "combined_correction_ratio",
        "task_conflict_ratio_pre",
        "task_conflict_ratio_post",
        "beta_E_local",
    ):
        steps[column] = pd.to_numeric(steps[column], errors="coerce")

    cap_by_parameter = {
        parameter: configuration.guard.policy_for(parameter).shape_max_ratio
        for parameter in ("fc1.weight", "fc2.weight")
    }
    steps["shape_cap"] = steps["parameter"].map(cap_by_parameter)
    steps["shape_cap_hit"] = (
        steps["shape_correction_ratio"]
        >= steps["shape_cap"] - 1e-6
    )

    saturation = (
        steps.groupby(["epoch", "parameter"], as_index=False)
        .agg(
            due_checks=("global_step", "size"),
            mean_shape_ratio=("shape_correction_ratio", "mean"),
            max_shape_ratio=("shape_correction_ratio", "max"),
            shape_cap_fraction=("shape_cap_hit", "mean"),
            mean_volume_ratio=("volume_correction_ratio", "mean"),
            mean_combined_ratio=("combined_correction_ratio", "mean"),
            mean_beta_E_local=("beta_E_local", "mean"),
            mean_task_conflict_pre=("task_conflict_ratio_pre", "mean"),
            mean_task_conflict_post=("task_conflict_ratio_post", "mean"),
        )
    )
    display(saturation.tail(18))

    for parameter in ("fc1.weight", "fc2.weight"):
        layer = saturation.loc[saturation["parameter"].eq(parameter)]
        fig, ax = plt.subplots(figsize=(10, 5), dpi=135)
        ax.plot(
            layer["epoch"],
            layer["shape_cap_fraction"],
            marker="o",
            label="Fraction hitting shape cap",
        )
        ax.set_ylim(-0.02, 1.02)
        ax.set(
            xlabel="Epoch",
            ylabel="Cap-hit fraction",
            title=f"{parameter}: shell-beta cap saturation",
        )
        ax.grid(alpha=0.25)
        ax.legend()
        plt.tight_layout()
        plt.show()

    post = steps["task_conflict_ratio_post"].dropna()
    print("Maximum post-safeguard task conflict:", post.max())
    print("Positive fraction after safeguard:", (post > 1e-6).mean())


## Compare at matched convergence

These plots distinguish a genuine generalization change from a simple delay in optimization.


In [ ]:
plot_controller(result.controller)
plot_corrections(result.correction_summary)
plot_matched_convergence(
    result.performance,
    result.weightwatcher,
)

ok = result.weightwatcher.loc[
    result.weightwatcher["status"].eq("ok")
].copy()
assert ok["alpha_source"].astype(str).eq("WeightWatcher").all()
assert ok["ERG_gap_source"].astype(str).eq("WeightWatcher").all()

display(
    ok[
        [
            "run",
            "epoch",
            "layer_name",
            "alpha",
            "ERG_gap",
            "beta_E_midpoint",
            "scale_balance_reliable",
        ]
    ].tail(18)
)


## Interpretation

A valid Stabilized V2 run must satisfy all of the following:

- the preflight prints `PREFLIGHT PASSED`;
- every epoch prints `STABILIZED V2 CHANNEL GATES FOR NEXT EPOCH`;
- FC1 and FC2 are never turned off solely for `low ECS confidence`;
- an enabled layer with \(\alpha\leq2.05\) has nonzero trace-log volume gain;
- low raw confidence may set the shape gain to zero without disabling the volume channel;
- FC1 shape corrections are capped at 2% of the AdamW step;
- FC2 shape corrections are capped at 0.75%;
- post-safeguard task conflict remains non-positive up to numerical tolerance.

If any of these conditions fails, the notebook raises an exception instead of continuing with a mislabeled V1 run.
